In [ ]:
#| default_exp facade

In [ ]:
#| export
"""A compact, dataframe-first public interface for interactive probe work."""
from __future__ import annotations

from dataclasses import replace
import json
from pathlib import Path
from typing import Iterable
import re

import pandas as pd

from probeharbour.bed import candidates_to_bed
from probeharbour.coordinates import coordinate_map_from_exons
from probeharbour.design import generate_junction_candidates, select_candidates
from probeharbour.models import Candidate, GenomicCoordinates, Provenance
from probeharbour.rules import RuleSet
from probeharbour.thermo import ThermoParameters, characterize as characterize_candidate

In [ ]:
#| export
def read_bed(bed: str | Path | Iterable[str | Path], *, reference_fasta: str | Path,
             name: str | None = None, panel_role: str | None = None,
             arm_length: int | None = None) -> "ProbeStore":
    """Create a :class:`ProbeStore` from 10x probe-location BED file(s).

    A Xenium locations BED contains genomic coordinates, not probe sequences.
    The matching reference FASTA is therefore required. BED12 blocks are
    concatenated before strand-aware sequence reconstruction, preserving probes
    that span exon-exon junctions. Sequence is split equally unless
    ``arm_length`` is supplied.

    BED coordinates are interpreted according to the BED specification:
    zero-based, half-open. Xenium BED strand is treated as the probe strand
    (opposite the targeted transcript). The FASTA must describe the same genome assembly.
    """
    paths = [Path(bed)] if isinstance(bed, (str, Path)) else [Path(path) for path in bed]
    if not paths:
        raise ValueError("At least one BED path is required")
    fasta = _IndexedFasta(reference_fasta)
    rows: list[dict] = []
    for path in paths:
        if not path.is_file():
            raise FileNotFoundError(f"BED file not found: {path}")
        with path.open() as handle:
            for line_number, line in enumerate(handle, start=1):
                if not line.strip() or line.startswith(("track", "browser", "#")):
                    continue
                fields = line.rstrip("\n").split("\t")
                if len(fields) < 6:
                    raise ValueError(f"{path}:{line_number}: expected at least BED6 columns")
                chrom, start_text, end_text, bed_name, score, strand = fields[:6]
                start, end = int(start_text), int(end_text)
                if strand not in {"+", "-"}:
                    raise ValueError(f"{path}:{line_number}: strand must be '+' or '-'")
                blocks = [(start, end)]
                if len(fields) >= 12:
                    count = int(fields[9])
                    sizes = [int(value) for value in fields[10].rstrip(",").split(",") if value]
                    offsets = [int(value) for value in fields[11].rstrip(",").split(",") if value]
                    if len(sizes) != count or len(offsets) != count:
                        raise ValueError(f"{path}:{line_number}: invalid BED12 block columns")
                    blocks = [(start + offset, start + offset + size)
                              for size, offset in zip(sizes, offsets)]
                genomic = "".join(fasta.fetch(chrom, block_start, block_end)
                                  for block_start, block_end in blocks).upper()
                # Xenium BED strand describes the physical probe strand. rbd5+rbd3
                # follows the targeted transcript sequence used throughout
                # ProbeHarbour, so it is the opposite orientation.
                binder = _reverse_complement(genomic) if strand == "+" else genomic
                split = arm_length if arm_length is not None else len(binder) // 2
                if split <= 0 or split >= len(binder):
                    raise ValueError(f"{path}:{line_number}: cannot split {len(binder)} nt into two arms")
                parsed = _parse_xenium_bed_name(bed_name)
                probe_name = parsed.pop("probe_name", f"probe_{len(rows) + 1}")
                rows.append({
                    "probe_name": probe_name, **parsed,
                    "rbd5": binder[:split], "rbd3": binder[split:],
                    "chrom": chrom, "chrom_start": start, "chrom_end": end,
                    "strand": strand, "bed_score": score,
                    "bed_block_count": len(blocks),
                    "bed_block_sizes": ",".join(str(block_end - block_start)
                                                for block_start, block_end in blocks),
                    "bed_block_starts": ",".join(str(block_start - start)
                                                 for block_start, _ in blocks),
                    "junction_spanning": len(blocks) > 1,
                    "source_bed": str(path),
                })
    panel_name = name or (paths[0].stem if len(paths) == 1 else "bed_panel")
    metadata = {
        "input_format": "bed", "source_beds": [str(path) for path in paths],
        "reference_fasta": str(Path(reference_fasta)),
        "coordinate_convention": "BED: zero-based, half-open",
        "sequence_derivation": "target-region sequence from BED blocks; reverse-complemented on BED '+' probe strand",
    }
    return ProbeStore(pd.DataFrame(rows), name=panel_name, source_path=paths[0],
                      metadata=metadata, panel_role=panel_role)

In [ ]:
#| export
def annotate_transcripts(store: "ProbeStore", transcript_data, *, inplace: bool = True) -> "ProbeStore":
    """Annotate BED-derived probes against an ``allos.TranscriptData`` object.

    All input rows are retained. Unmapped probes receive ``mapping_status`` and
    ``mapping_reason`` values instead of disappearing from the resulting store.
    Only public TranscriptData methods are used.
    """
    result = store if inplace else store.copy()
    required = {"chrom", "chrom_start", "chrom_end", "strand"}
    missing = required - set(result.obs)
    if missing:
        raise ValueError(f"Transcript annotation requires BED coordinate columns: {sorted(missing)}")

    annotations = []
    gene_cache: dict[tuple[str, str], list[str]] = {}
    transcript_cache: dict[str, tuple[str, str, list[tuple[int, int]]]] = {}
    for _, row in result.obs.iterrows():
        gene_name = str(row.get("gene_name", "") or "")
        gene_id = str(row.get("gene_id", "") or "")
        cache_key = (gene_name, gene_id)
        if cache_key not in gene_cache:
            transcript_ids = []
            if gene_id and gene_id.lower() != "nan":
                transcript_ids = list(transcript_data.get_transcripts_by_gene_id(gene_id))
            if not transcript_ids and gene_name and gene_name.lower() != "nan":
                transcript_ids = list(transcript_data.get_transcripts_by_gene_name(gene_name))
            gene_cache[cache_key] = transcript_ids
        transcript_ids = gene_cache[cache_key]
        blocks = _row_blocks(row)
        supporting = []
        support_maps = {}
        for transcript_id in transcript_ids:
            if transcript_id not in transcript_cache:
                exon_coords, strand_code = transcript_data.get_exon_coords_and_strand(transcript_id)
                info = transcript_data.get_transcript_info(transcript_id)
                chrom = str(info.get("chromosome", transcript_data.get_chromosome(transcript_id)))
                strand = "+" if strand_code in {1, "+"} else "-"
                exons = sorted((min(pair), max(pair)) for pair in exon_coords)
                transcript_cache[transcript_id] = (chrom, strand, exons)
            transcript_chrom, transcript_strand, exons = transcript_cache[transcript_id]
            target_strand = "-" if row["strand"] == "+" else "+"
            if _norm_chrom(transcript_chrom) != _norm_chrom(row["chrom"]) or transcript_strand != target_strand:
                continue
            exon_indices = []
            for block_start, block_end in blocks:
                containing = next((index for index, (exon_start, exon_end) in enumerate(exons)
                                   if exon_start <= block_start and block_end <= exon_end), None)
                if containing is None:
                    break
                exon_indices.append(containing)
            else:
                if len(exon_indices) == 1 or all(right == left + 1 for left, right in zip(exon_indices, exon_indices[1:])):
                    supporting.append(transcript_id)
                    support_maps[transcript_id] = exons

        if not transcript_ids:
            annotations.append(_unmapped_annotation("gene_not_found"))
            continue
        if not supporting:
            annotations.append(_unmapped_annotation("interval_not_supported_by_gene_transcripts",
                                                    transcripts_considered=len(transcript_ids)))
            continue

        primary = supporting[0]
        probe_strand = str(row["strand"])
        target_strand = "-" if probe_strand == "+" else "+"
        block_lengths = [end - start for start, end in blocks]
        binder_junction_positions = []
        if len(blocks) > 1:
            ordered_lengths = block_lengths if probe_strand == "+" else list(reversed(block_lengths))
            cumulative = 0
            for length in ordered_lengths[:-1]:
                cumulative += length
                binder_junction_positions.append(cumulative)
        split = len(str(row["rbd5"]))
        junction_ids = []
        for left, right in zip(blocks, blocks[1:]):
            junction_ids.append(f"{_norm_chrom(row['chrom'])}:{left[1]}-{right[0]}:{target_strand}")
        annotations.append({
            "mapping_status": "mapped",
            "mapping_reason": "",
            "target_strand": target_strand,
            "transcript_id": primary,
            "supporting_transcripts": supporting,
            "n_supporting_transcripts": len(supporting),
            "transcripts_considered": len(transcript_ids),
            "target_coverage": len(supporting) / len(transcript_ids),
            "junction_id": junction_ids[0] if len(junction_ids) == 1 else ";".join(junction_ids),
            "on_junction": split in binder_junction_positions,
            "junction_spanning": len(blocks) > 1,
            "isoform_specific_junction": len(blocks) > 1 and len(supporting) < len(transcript_ids),
        })

    frame = pd.DataFrame(annotations, index=result.obs.index)
    for column in frame:
        result.obs[column] = frame[column]
    result.metadata["transcript_annotation"] = {
        "method": "public-allos-exon-containment-v1",
        "rows": len(result.obs),
        "mapped": int(result.obs["mapping_status"].eq("mapped").sum()),
        "unmapped": int(result.obs["mapping_status"].ne("mapped").sum()),
    }
    return result

In [ ]:
#| export
def map_probes(store: "ProbeStore", transcript_data, *, inplace: bool = True) -> "ProbeStore":
    """Map sequence-only probes to transcripts and actual genomic coordinates.

    ``rbd5 + rbd3`` is searched exactly in every annotated transcript for the
    row's gene. Coordinates are calculated from the matched transcript bases;
    shifted probes are never snapped to a nearby junction. All rows are kept.
    """
    result = store if inplace else store.copy()
    gene_column = _first_column(result.obs, "gene_name", "gene", "gene_id")
    if gene_column is None:
        raise ValueError("Sequence mapping requires a gene_name, gene, or gene_id column")
    missing = {"rbd5", "rbd3"} - set(result.obs)
    if missing:
        raise ValueError(f"Sequence mapping requires columns: {sorted(missing)}")

    transcript_cache = {}
    gene_cache = {}
    annotations = []
    for _, row in result.obs.iterrows():
        gene = str(row[gene_column])
        cache_key = (gene_column, gene)
        if cache_key not in gene_cache:
            if gene_column == "gene_id":
                tids = list(transcript_data.get_transcripts_by_gene_id(gene))
            else:
                tids = list(transcript_data.get_transcripts_by_gene_name(gene))
                if not tids:
                    tids = list(transcript_data.get_transcripts_by_gene_id(gene))
            gene_cache[cache_key] = tids
        transcript_ids = gene_cache[cache_key]
        binder = f"{row['rbd5']}{row['rbd3']}".upper()
        arm_length = len(str(row["rbd5"]))
        matches = []
        for transcript_id in transcript_ids:
            if transcript_id not in transcript_cache:
                sequence = str(transcript_data.get_transcript_sequence(transcript_id)).upper()
                exon_coords, strand_code = transcript_data.get_exon_coords_and_strand(transcript_id)
                info = transcript_data.get_transcript_info(transcript_id)
                strand = "+" if strand_code in {1, "+"} else "-"
                cmap = coordinate_map_from_exons(
                    transcript_id, str(info["chromosome"]), strand,
                    [(min(pair), max(pair)) for pair in exon_coords],
                )
                transcript_cache[transcript_id] = (sequence, cmap)
            sequence, cmap = transcript_cache[transcript_id]
            position = sequence.find(binder)
            while position >= 0:
                cut = position + arm_length
                spanned = [junction for junction in cmap.junction_positions
                           if position < junction < position + len(binder)]
                nearest = min(cmap.junction_positions, key=lambda junction: abs(cut - junction)) if cmap.junction_positions else None
                matches.append((
                    not bool(spanned), abs(cut - nearest) if nearest is not None else len(sequence),
                    transcript_id, position, cut, spanned, nearest, cmap,
                ))
                position = sequence.find(binder, position + 1)

        if not matches:
            reason = "gene_not_found" if not transcript_ids else "sequence_not_found_in_gene_transcripts"
            annotations.append({
                "mapping_status": "unmapped", "mapping_reason": reason,
                "transcript_id": None, "supporting_transcripts": [],
                "n_supporting_transcripts": 0, "n_sequence_matches": 0,
            })
            continue
        matches.sort(key=lambda item: (item[0], item[1], item[2], item[3]))
        _, _, transcript_id, position, cut, spanned, nearest, cmap = matches[0]
        supporting = sorted({match[2] for match in matches})
        rbd5_first = cmap.spliced_to_genomic(position)
        rbd5_last = cmap.spliced_to_genomic(cut - 1)
        rbd3_first = cmap.spliced_to_genomic(cut)
        rbd3_last = cmap.spliced_to_genomic(position + len(binder) - 1)
        all_bases = (rbd5_first, rbd5_last, rbd3_first, rbd3_last)
        junction_ids = []
        for junction in spanned:
            left = cmap.spliced_to_genomic(junction - 1)
            right = cmap.spliced_to_genomic(junction)
            junction_ids.append(f"{cmap.chrom}:{min(left, right) + 1}-{max(left, right)}:{cmap.strand}")
        junction_supporting = []
        if junction_ids:
            target_junction = junction_ids[0]
            for candidate_transcript in transcript_ids:
                _, candidate_map = transcript_cache[candidate_transcript]
                annotated_junctions = set()
                for junction in candidate_map.junction_positions:
                    left = candidate_map.spliced_to_genomic(junction - 1)
                    right = candidate_map.spliced_to_genomic(junction)
                    annotated_junctions.add(
                        f"{candidate_map.chrom}:{min(left, right) + 1}-{max(left, right)}:{candidate_map.strand}"
                    )
                if target_junction in annotated_junctions:
                    junction_supporting.append(candidate_transcript)
        junction_count = len(junction_supporting)
        junction_class = None
        if junction_ids:
            junction_class = ("specific" if junction_count == 1 else
                              ("constitutive" if junction_count == len(transcript_ids)
                               else "partially specific"))
        annotations.append({
            "mapping_status": "mapped", "mapping_reason": "",
            "transcript_id": transcript_id, "supporting_transcripts": supporting,
            "n_supporting_transcripts": len(supporting), "n_sequence_matches": len(matches),
            "gene_transcript_count": len(transcript_ids),
            "junction_supporting_transcripts": sorted(junction_supporting),
            "junction_transcript_count": junction_count,
            "target_coverage": 100 * junction_count / len(transcript_ids),
            "junction_specificity_class": junction_class,
            "mapped_cut": cut, "junction_id": ";".join(junction_ids),
            "on_junction": nearest is not None and cut == nearest,
            "junction_spanning": bool(spanned),
            "junction_shift": cut - nearest if nearest is not None else pd.NA,
            "isoform_specific_junction": bool(spanned) and len(supporting) < len(transcript_ids),
            "chrom": cmap.chrom, "strand": cmap.strand,
            "chrom_start": min(all_bases), "chrom_end": max(all_bases) + 1,
            "rbd5_chrom_start": min(rbd5_first, rbd5_last),
            "rbd5_chrom_end": max(rbd5_first, rbd5_last) + 1,
            "rbd3_chrom_start": min(rbd3_first, rbd3_last),
            "rbd3_chrom_end": max(rbd3_first, rbd3_last) + 1,
        })
    frame = pd.DataFrame(annotations, index=result.obs.index)
    for column in frame:
        result.obs[column] = frame[column]
    result.metadata["sequence_mapping"] = {
        "method": "exact-spliced-transcript-search-v1",
        "rows": len(result.obs),
        "mapped": int(result.obs["mapping_status"].eq("mapped").sum()),
        "unmapped": int(result.obs["mapping_status"].ne("mapped").sum()),
        "coordinate_policy": "actual matched sequence; never junction-snapped",
    }
    return result

In [ ]:
#| export
def build_store_from_bed(bed: str | Path | Iterable[str | Path], *, reference_fasta: str | Path,
                         transcript_data=None, name: str | None = None,
                         panel_role: str | None = "existing", characterize: bool = True,
                         chemistry: str | None = None, genes: Iterable[str] | None = None,
                         n_genes: int | None = None, seed: int = 42) -> "ProbeStore":
    """Build a BED-derived ProbeStore with optional transcript annotation.

    This explicit pipeline replaces the legacy builder's silent row dropping:
    read BED + FASTA, optionally annotate every row with TranscriptData, then
    characterize and optionally attach a prospective chemistry score.
    """
    store = read_bed(bed, reference_fasta=reference_fasta, name=name, panel_role=panel_role)
    if genes is not None and n_genes is not None:
        raise ValueError("Pass genes or n_genes, not both")
    gene_column = _first_column(store.obs, "gene_name", "gene_id")
    selected_genes = None
    if genes is not None:
        selected_genes = [str(gene) for gene in genes]
    elif n_genes is not None:
        if n_genes < 1:
            raise ValueError("n_genes must be positive")
        if gene_column is None:
            raise ValueError("n_genes requires gene names or IDs parsed from BED")
        available = pd.Series(store.obs[gene_column].dropna().astype(str).unique())
        selected_genes = available.sample(n=min(n_genes, len(available)), random_state=seed).tolist()
    if selected_genes is not None:
        if gene_column is None:
            raise ValueError("genes requires gene names or IDs parsed from BED")
        store.obs = store.obs.loc[store.obs[gene_column].astype(str).isin(selected_genes)].reset_index(drop=True)
        store.metadata["build_subset"] = {
            "gene_column": gene_column, "genes": selected_genes,
            "requested_n_genes": n_genes, "seed": seed if n_genes is not None else None,
        }
    if transcript_data is not None:
        annotate_transcripts(store, transcript_data)
    if characterize:
        store.characterize()
    if chemistry is not None:
        store.annotate_scores(chemistry, score_column="prospective_score")
    store.metadata["build_pipeline"] = [
        "read_bed", *( ["annotate_transcripts"] if transcript_data is not None else []),
        *( ["characterize"] if characterize else []),
        *( [f"score:{chemistry}"] if chemistry is not None else []),
    ]
    return store


def _row_blocks(row) -> list[tuple[int, int]]:
    start, end = int(row["chrom_start"]), int(row["chrom_end"])
    sizes_text, starts_text = row.get("bed_block_sizes"), row.get("bed_block_starts")
    if pd.isna(sizes_text) or pd.isna(starts_text):
        return [(start, end)]
    sizes = [int(value) for value in str(sizes_text).split(",") if value]
    offsets = [int(value) for value in str(starts_text).split(",") if value]
    return [(start + offset, start + offset + size) for size, offset in zip(sizes, offsets)]


def _norm_chrom(value) -> str:
    return str(value).removeprefix("chr")


def _unmapped_annotation(reason: str, *, transcripts_considered: int = 0) -> dict:
    return {
        "mapping_status": "unmapped", "mapping_reason": reason,
        "target_strand": None,
        "transcript_id": None, "supporting_transcripts": [],
        "n_supporting_transcripts": 0, "transcripts_considered": transcripts_considered,
        "target_coverage": 0.0, "junction_id": "", "on_junction": False,
        "junction_spanning": False, "isoform_specific_junction": False,
    }


def _parse_xenium_bed_name(value: str) -> dict:
    match = re.match(r"^(.+?) \(([^)]+)\) \| ([^|]+) \| ([^|]+)$", value)
    if not match:
        safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", value).strip("_") or "probe"
        return {"probe_name": safe, "bed_name": value}
    gene_name, gene_id, codeword, probeset = (part.strip() for part in match.groups())
    return {
        "probe_name": f"panel_{gene_name}_cw{codeword}_ps{probeset}",
        "gene_name": gene_name, "gene_id": gene_id,
        "codeword": int(codeword) if codeword.isdigit() else codeword,
        "probeset": int(probeset) if probeset.isdigit() else probeset,
        "bed_name": value,
    }


def _reverse_complement(sequence: str) -> str:
    return sequence.translate(str.maketrans("ACGTNacgtn", "TGCANtgcan"))[::-1]


class _IndexedFasta:
    """Minimal reader for an uncompressed FASTA with a samtools ``.fai`` index."""

    def __init__(self, path: str | Path):
        self.path = Path(path)
        index_path = Path(f"{self.path}.fai")
        if not self.path.is_file():
            raise FileNotFoundError(f"Reference FASTA not found: {self.path}")
        if self.path.suffix == ".gz":
            raise ValueError("read_bed requires an indexed, uncompressed FASTA; pass the .fa path")
        if not index_path.is_file():
            raise FileNotFoundError(f"FASTA index not found: {index_path}; create it with samtools faidx")
        self.index = {}
        for line in index_path.read_text().splitlines():
            contig, length, offset, line_bases, line_width = line.split("\t")[:5]
            self.index[contig] = tuple(map(int, (length, offset, line_bases, line_width)))

    def fetch(self, chrom: str, start: int, end: int) -> str:
        candidates = (chrom, chrom.removeprefix("chr"), f"chr{chrom}")
        contig = next((candidate for candidate in candidates if candidate in self.index), None)
        if contig is None:
            raise KeyError(f"Chromosome {chrom!r} is absent from {self.path}")
        length, offset, line_bases, line_width = self.index[contig]
        if start < 0 or end < start or end > length:
            raise ValueError(f"Invalid interval {chrom}:{start}-{end} for contig length {length}")
        byte_start = offset + (start // line_bases) * line_width + start % line_bases
        byte_end = offset + ((end - 1) // line_bases) * line_width + (end - 1) % line_bases + 1 if end > start else byte_start
        with self.path.open("rb") as handle:
            handle.seek(byte_start)
            return handle.read(byte_end - byte_start).replace(b"\n", b"").replace(b"\r", b"").decode()[:end-start]

In [ ]:
#| export
class ProbeStore:
    """A collection of probes with a Scanpy-like dataframe at ``.obs``.

    Methods that add annotations operate in place and return ``self`` so a
    notebook can either inspect each step or chain a short workflow.
    """

    def __init__(self, obs: pd.DataFrame, *, name: str = "probes", source_path: str | Path | None = None,
                 candidates: Iterable[Candidate] | None = None, metadata: dict | None = None,
                 panel_role: str | None = None):
        self.name = name
        self.obs = obs.copy()
        if panel_role is not None:
            self.obs["panel_role"] = str(panel_role)
        self.source_path = Path(source_path) if source_path is not None else None
        self._candidates = list(candidates) if candidates is not None else None
        self.metadata = dict(metadata or {})

    @property
    def candidates(self) -> pd.DataFrame:
        """Compatibility alias for the observation dataframe."""
        return self.obs

    def __repr__(self) -> str:
        return f"ProbeStore(name={self.name!r}, n_probes={len(self.obs)}, n_columns={len(self.obs.columns)})"

    def summary(self) -> pd.DataFrame:
        gene_col = _first_column(self.obs, "gene", "gene_name", "gene_id")
        preferred_scores = ["raw_design_score", "stored_raw_design_score", "recalculated_score", "score"]
        score_candidates = preferred_scores + [
            column for column in self.obs
            if column.endswith("_score") and column not in preferred_scores
            and column != "bed_score"
        ]
        score_col = next(
            (column for column in score_candidates
             if column in self.obs and self.obs[column].notna().any()),
            None,
        )
        return pd.DataFrame([{
            "panel": self.name,
            "probes": len(self.obs),
            "genes": self.obs[gene_col].nunique(dropna=True) if gene_col else 0,
            "median_score": pd.to_numeric(self.obs[score_col], errors="coerce").median()
            if score_col else float("nan"),
            "score_column": score_col,
        }])

    def copy(self) -> "ProbeStore":
        """Return an independent in-memory copy."""
        return ProbeStore(self.obs, name=self.name, source_path=self.source_path,
                          candidates=self._candidates, metadata=self.metadata)

    def query(self, expression: str, **kwargs) -> "ProbeStore":
        """Return a filtered view using :meth:`pandas.DataFrame.query` syntax."""
        selected = self.obs.query(expression, **kwargs).copy()
        metadata = dict(self.metadata)
        metadata.setdefault("queries", []).append(expression)
        return ProbeStore(selected, name=self.name, source_path=self.source_path, metadata=metadata)

    def characterize(self, parameters: ThermoParameters = ThermoParameters()) -> "ProbeStore":
        """Calculate sequence and Primer3 properties without replacing stored scores."""
        self._ensure_candidate_ids()
        candidates = self.to_candidates()
        ids = self._candidate_ids()
        candidate_lookup = {candidate.candidate_id: candidate for candidate in candidates}
        if "candidate_id" not in self.obs:
            self.obs["candidate_id"] = ids.values
        self.obs["binder"] = ids.map(lambda value: candidate_lookup[value].binder)
        self.obs["arm_length"] = ids.map(lambda value: candidate_lookup[value].arm_length)
        self.obs["lig_dinuc"] = ids.map(lambda value: candidate_lookup[value].lig_dinuc)
        values = [characterize_candidate(candidate, parameters).as_dict() for candidate in candidates]
        frame = pd.DataFrame(values).set_index("candidate_id")
        for column in frame.columns:
            self.obs[column] = ids.map(frame[column])
        return self

    def annotate_transcripts(self, transcript_data) -> "ProbeStore":
        """Map BED-derived coordinates to transcripts without dropping failures."""
        return annotate_transcripts(self, transcript_data, inplace=True)

    def map_probes(self, transcript_data) -> "ProbeStore":
        """Map sequence-only probes onto annotated transcripts and coordinates."""
        return map_probes(self, transcript_data, inplace=True)

    def explain(self, chemistry: str = "prime_5k") -> pd.DataFrame:
        """Explain recalculated scores alongside immutable historic scores."""
        self._ensure_candidate_ids()
        ruleset = RuleSet.from_packaged(chemistry)
        explanations = []
        for _, row in self.obs.iterrows():
            explanation = ruleset.explain(row.to_dict()).as_dict()
            explanation["candidate_id"] = _row_id(row)
            explanation["stored_raw_design_score"] = row.get("stored_raw_design_score", row.get("raw_design_score"))
            explanations.append(explanation)
        return pd.DataFrame(explanations)

    def annotate_scores(self, chemistry: str = "prime_5k", *, score_column: str = "recalculated_score",
                        overwrite: bool = False) -> "ProbeStore":
        """Attach a prospective score without overwriting historic score columns."""
        if score_column in self.obs and not overwrite:
            raise ValueError(f"Column already exists: {score_column}; pass overwrite=True explicitly")
        explanation = self.explain(chemistry).set_index("candidate_id")
        ids = self._candidate_ids()
        self.obs[score_column] = ids.map(explanation["recalculated_score"])
        self.obs[f"{score_column}_modifiers"] = ids.map(explanation["modifiers"])
        self.obs[f"{score_column}_base_score"] = ids.map(explanation["base_score"])
        self.obs[f"{score_column}_ruleset_id"] = ids.map(explanation["ruleset_id"])
        self.metadata.setdefault("score_annotations", []).append({
            "column": score_column, "chemistry": chemistry,
            "ruleset_id": explanation["ruleset_id"].iloc[0] if len(explanation) else None,
        })
        return self

    def check_panel_compatibility(self, reference: "ProbeStore | pd.DataFrame", **kwargs) -> "ProbeStore":
        """Annotate compatibility with an existing panel without changing scores.

        Parameters are forwarded to
        :func:`probeharbour.panel_compatibility.check_panel_compatibility`.
        """
        from probeharbour.panel_compatibility import check_panel_compatibility

        annotations = check_panel_compatibility(self, reference, **kwargs).set_index("candidate_id")
        ids = (self.obs["panel_record_id"].astype(str)
               if "panel_record_id" in self.obs else self._candidate_ids())
        for column in annotations.columns:
            self.obs[column] = ids.map(annotations[column])
        self.obs["existing_panel_compatibility_pass"] = self.obs["panel_compatibility_pass"]
        self.obs["existing_panel_compatibility_reasons"] = self.obs["panel_compatibility_reasons"]
        self.metadata.setdefault("panel_compatibility", []).append({
            "reference_panel": getattr(reference, "name", "dataframe"),
            "complement_run_threshold": kwargs.get("complement_run_threshold", 12),
            "heterodimer_dg_threshold": kwargs.get("heterodimer_dg_threshold", -10.5),
            "calculate_heterodimer": kwargs.get("calculate_heterodimer", True),
        })
        return self

    def check_internal_compatibility(self, *, min_distance: int = 75,
                                     role_column: str = "panel_role", **kwargs) -> "ProbeStore":
        """Audit self-excluding sequence conflicts and spacing in a combined panel."""
        from probeharbour.panel_compatibility import check_internal_compatibility

        record_ids = self.obs.get("panel_record_id")
        if (record_ids is None or record_ids.isna().any()
                or record_ids.astype(str).duplicated().any()):
            ids = self._candidate_ids()
            self.obs["panel_record_id"] = [f"{value}:{index}" for index, value in enumerate(ids)]
        annotations = check_internal_compatibility(
            self, min_distance=min_distance, role_column=role_column, **kwargs
        ).set_index("panel_record_id")
        keys = self.obs["panel_record_id"].astype(str)
        for column in annotations.columns:
            self.obs[column] = keys.map(annotations[column])
        self.metadata.setdefault("internal_panel_compatibility", []).append({
            "min_distance": min_distance,
            "role_column": role_column,
            "complement_run_threshold": kwargs.get("complement_run_threshold", 12),
            "heterodimer_dg_threshold": kwargs.get("heterodimer_dg_threshold", -10.5),
        })
        return self

    def select(self, n_best: int = 1) -> pd.DataFrame:
        """Return auditable selection decisions; retain a decision for every probe."""
        rows = self.obs.copy()
        if "candidate_id" not in rows:
            rows["candidate_id"] = self._candidate_ids().values
        if "recalculated_score" not in rows:
            score_column = _first_column(rows, "raw_design_score", "stored_raw_design_score", "score")
            if score_column:
                rows["recalculated_score"] = rows[score_column]
        decisions = select_candidates(rows.to_dict("records"), limit=n_best)
        return pd.DataFrame([decision.__dict__ for decision in decisions])

    def write_fasta(self, path: str | Path) -> Path:
        """Write concatenated binding arms as FASTA."""
        destination = Path(path)
        lines = []
        for candidate in self.to_candidates():
            lines.extend((f">{candidate.candidate_id}", candidate.binder))
        destination.write_text("\n".join(lines) + ("\n" if lines else ""))
        return destination

    def write_bed(self, path: str | Path) -> Path:
        """Write candidates with genomic coordinates as BED."""
        destination = Path(path)
        destination.write_text(candidates_to_bed(self.to_candidates()))
        return destination

    def write_csv(self, path: str | Path, *, index: bool = False) -> Path:
        """Write the complete observation table as CSV."""
        destination = Path(path)
        self.obs.to_csv(destination, index=index)
        return destination

    def save(self, path: str | Path) -> Path:
        """Persist a portable ProbeStore directory with Parquet and metadata."""
        destination = Path(path)
        destination.mkdir(parents=True, exist_ok=True)
        observations = self.obs.copy()
        # PyArrow cannot represent a struct column whose dictionaries are all
        # empty (for example ``extra={}`` on newly designed candidates).
        # Empty optional metadata is semantically missing, so persist it as
        # null while preserving non-empty dictionaries and the in-memory store.
        for column in observations.select_dtypes(include="object"):
            if observations[column].map(lambda value: isinstance(value, dict)).any():
                observations[column] = observations[column].map(
                    lambda value: None if isinstance(value, dict) and not value else value
                )
        observations.to_parquet(destination / "probes.parquet", index=False)
        metadata = dict(self.metadata)
        metadata.update({"name": self.name, "n_probes": len(self.obs), "format_version": "1"})
        (destination / "metadata.json").write_text(json.dumps(metadata, indent=2, sort_keys=True) + "\n")
        self.source_path = destination / "probes.parquet"
        return destination

    @classmethod
    def load(cls, path: str | Path) -> "ProbeStore":
        """Load a directory written by :meth:`save` or a legacy ProbeStore."""
        source = Path(path)
        parquet = source / "probes.parquet" if source.is_dir() else source
        metadata_path = parquet.parent / "metadata.json"
        metadata = json.loads(metadata_path.read_text()) if metadata_path.is_file() else {}
        return cls(pd.read_parquet(parquet), name=metadata.get("name", parquet.parent.name),
                   source_path=parquet, metadata=metadata)

    @classmethod
    def merge(cls, stores: Iterable["ProbeStore"], *, name: str = "merged",
              roles: Iterable[str] | None = None) -> "ProbeStore":
        """Concatenate stores while retaining source and optional panel roles."""
        items = list(stores)
        if not items:
            raise ValueError("At least one ProbeStore is required")
        if roles is not None:
            role_values = list(roles)
        else:
            role_values = []
            for store in items:
                existing_roles = store.obs["panel_role"].dropna().astype(str).unique() if "panel_role" in store.obs else []
                role_values.append(existing_roles[0] if len(existing_roles) == 1 else store.name)
        if len(role_values) != len(items):
            raise ValueError("roles must contain exactly one value per store")
        frames = []
        for store, role in zip(items, role_values):
            frame = store.obs.copy()
            id_column = _first_column(frame, "candidate_id", "probe_name")
            ids = frame[id_column].astype(str).tolist() if id_column else [f"probe-{i + 1}" for i in range(len(frame))]
            frame["source_store"] = store.name
            frame["panel_role"] = str(role)
            frame["panel_record_id"] = [f"{role}:{candidate_id}:{index}" for index, candidate_id in enumerate(ids)]
            frames.append(frame)
        obs = pd.concat(frames, ignore_index=True)
        return cls(obs, name=name, metadata={
            "merged_from": [store.name for store in items], "panel_roles": role_values,
            "source_stores": [
                {
                    "name": store.name,
                    "source_path": str(store.source_path) if store.source_path is not None else None,
                    "metadata": dict(store.metadata),
                }
                for store in items
            ],
        })

    def to_candidates(self) -> list[Candidate]:
        if self._candidates is not None:
            return self._candidates
        return [_candidate_from_row(row, self.name) for _, row in self.obs.iterrows()]

    def _candidate_ids(self) -> pd.Series:
        column = _first_column(self.obs, "candidate_id", "probe_name")
        if column:
            return self.obs[column].astype(str)
        return pd.Series([f"probe-{i + 1}" for i in range(len(self.obs))], index=self.obs.index)

    def _ensure_candidate_ids(self) -> None:
        """Attach stable row-unique IDs without altering user-facing probe names."""
        if "candidate_id" in self.obs and self.obs["candidate_id"].notna().all() \
                and not self.obs["candidate_id"].astype(str).duplicated().any():
            return
        display_column = _first_column(self.obs, "probe_name", "name")
        displays = (self.obs[display_column].astype(str) if display_column
                    else pd.Series("probe", index=self.obs.index))
        counts = displays.groupby(displays, sort=False).cumcount()
        totals = displays.map(displays.value_counts())
        self.obs["candidate_id"] = [
            display if total == 1 else f"{display}:row:{position + 1}"
            for display, position, total in zip(displays, counts, totals)
        ]

In [ ]:
#| export
def load_panel(path: str | Path, *, name: str | None = None,
               column_map: dict[str, str] | None = None,
               panel_role: str | None = None) -> ProbeStore:
    """Load a ProbeStore directory, Parquet file, or probe CSV.

    ``column_map`` normalizes external schemas while preserving unmapped source
    columns. ``panel_role`` belongs to store context, not the input table.
    """
    source = Path(path).expanduser()
    data_path = source / "probes.parquet" if source.is_dir() else source
    if not data_path.is_file():
        raise FileNotFoundError(f"Panel file not found: {data_path}")
    if data_path.suffix.lower() == ".csv":
        observations = pd.read_csv(data_path)
    elif data_path.suffix.lower() in {".parquet", ".pq"}:
        observations = pd.read_parquet(data_path)
    else:
        raise ValueError(f"Unsupported panel format: {data_path.suffix}; expected CSV or Parquet")
    if column_map:
        unknown = set(column_map) - set(observations.columns)
        if unknown:
            raise ValueError(f"column_map references missing columns: {sorted(unknown)}")
        observations = observations.rename(columns=column_map)
    missing = {"rbd5", "rbd3"} - set(observations.columns)
    if missing:
        raise ValueError(
            f"Panel is missing required columns: {sorted(missing)}; use column_map for external schemas"
        )
    panel_name = name or (source.name if source.is_dir() else data_path.stem)
    metadata_path = data_path.parent / "metadata.json" if source.is_dir() else None
    metadata = json.loads(metadata_path.read_text()) if metadata_path and metadata_path.is_file() else {}
    metadata.update({"column_map": dict(column_map or {}), "input_format": data_path.suffix.lower()})
    return ProbeStore(observations, name=panel_name, source_path=data_path,
                      metadata=metadata, panel_role=panel_role)

In [ ]:
#| export
def design_junction(sequence: str, junction_position: int, *, chemistry: str = "prime_5k",
                    arm_length: int = 16, slides=range(-5, 6), candidate_prefix: str = "candidate",
                    dataset_id: str = "interactive", junction_id: str | None = None) -> ProbeStore:
    """Design every in-bounds slide around one transcript junction."""
    ruleset = RuleSet.from_packaged(chemistry)
    candidates = generate_junction_candidates(
        sequence, junction_position=junction_position, arm_length=arm_length, slides=slides,
        provenance=Provenance("designed", dataset_id, ruleset_id=ruleset.ruleset_id),
        candidate_prefix=candidate_prefix, junction_id=junction_id,
    )
    return _store_from_candidates(candidates, name=candidate_prefix)

In [ ]:
#| export
def design_gene(gene: str, transcript_data, *, chemistry: str = "prime_5k", arm_length: int = 16,
                slides=range(-5, 6), dataset_id: str = "interactive") -> ProbeStore:
    """Design annotated junctions for a gene from an ``allos.TranscriptData`` object.

    The adapter uses public TranscriptData methods and requires it to have been
    constructed with a reference FASTA.
    """
    transcript_ids = list(transcript_data.get_transcripts_by_gene_name(gene))
    if not transcript_ids:
        raise KeyError(f"Gene not found in annotation: {gene}")
    ruleset = RuleSet.from_packaged(chemistry)
    provenance = Provenance("designed", dataset_id, ruleset_id=ruleset.ruleset_id)
    candidates: list[Candidate] = []
    seen: set[tuple[str, str, str]] = set()
    junction_support: dict[str, set[str]] = {}
    for transcript_id in transcript_ids:
        sequence = str(transcript_data.get_transcript_sequence(transcript_id)).upper()
        exon_coords, strand_code = transcript_data.get_exon_coords_and_strand(transcript_id)
        exons = [(min(pair), max(pair)) for pair in exon_coords]
        info = transcript_data.get_transcript_info(transcript_id)
        strand = "+" if strand_code in {1, "+"} else "-"
        coordinate_map = coordinate_map_from_exons(transcript_id, str(info["chromosome"]), strand, exons)
        for junction_number, junction_position in enumerate(coordinate_map.junction_positions, start=1):
            left = coordinate_map.spliced_to_genomic(junction_position - 1)
            right = coordinate_map.spliced_to_genomic(junction_position)
            junction_id = f"{info['chromosome']}:{min(left, right) + 1}-{max(left, right)}:{strand}"
            junction_support.setdefault(junction_id, set()).add(str(transcript_id))
            generated = generate_junction_candidates(
                sequence, junction_position=junction_position, arm_length=arm_length, slides=slides,
                provenance=provenance, candidate_prefix=f"{gene}:{transcript_id}:J{junction_number}",
                junction_id=junction_id,
            )
            for candidate in generated:
                key = (candidate.rbd5, candidate.rbd3, junction_id)
                if key in seen:
                    continue
                seen.add(key)
                start_base = coordinate_map.spliced_to_genomic(candidate.ligation_site - arm_length)
                end_base = coordinate_map.spliced_to_genomic(candidate.ligation_site + arm_length - 1)
                rbd5_first = coordinate_map.spliced_to_genomic(candidate.ligation_site - arm_length)
                rbd5_last = coordinate_map.spliced_to_genomic(candidate.ligation_site - 1)
                rbd3_first = coordinate_map.spliced_to_genomic(candidate.ligation_site)
                rbd3_last = coordinate_map.spliced_to_genomic(candidate.ligation_site + arm_length - 1)
                coordinates = GenomicCoordinates(
                    str(info["chromosome"]), min(start_base, end_base), max(start_base, end_base) + 1, strand,
                    rbd5_start=min(rbd5_first, rbd5_last), rbd5_end=max(rbd5_first, rbd5_last) + 1,
                    rbd3_start=min(rbd3_first, rbd3_last), rbd3_end=max(rbd3_first, rbd3_last) + 1,
                )
                candidates.append(replace(candidate, gene_id=gene, transcript_id=transcript_id, coordinates=coordinates))
    store = _store_from_candidates(candidates, name=f"{gene}_design")
    # ``gene`` is the user-facing annotation symbol used to initiate design.
    # Keep it explicitly as ``gene_name`` as well as in Candidate.gene_id so
    # downstream tables, plots and exports do not have to infer it from IDs.
    store.obs["gene_name"] = gene
    total_transcripts = len(transcript_ids)
    support_count = store.obs["junction_id"].map(lambda value: len(junction_support.get(value, ())))
    store.obs["junction_supporting_transcripts"] = store.obs["junction_id"].map(
        lambda value: sorted(junction_support.get(value, ()))
    )
    store.obs["junction_transcript_count"] = support_count
    store.obs["gene_transcript_count"] = total_transcripts
    store.obs["target_coverage"] = support_count.div(total_transcripts).mul(100)
    store.obs["junction_specificity_class"] = support_count.map(
        lambda count: "specific" if count == 1 else (
            "constitutive" if count == total_transcripts else "partially specific"
        )
    )
    return store

In [ ]:
#| export
def _store_from_candidates(candidates: list[Candidate], *, name: str) -> ProbeStore:
    return ProbeStore(pd.DataFrame([candidate.as_dict() for candidate in candidates]), name=name, candidates=candidates)


def _first_column(frame: pd.DataFrame, *names: str) -> str | None:
    return next((name for name in names if name in frame), None)


def _row_id(row: pd.Series) -> str:
    return str(row.get("candidate_id", row.get("probe_name", row.name)))


def _candidate_from_row(row: pd.Series, dataset_id: str) -> Candidate:
    coordinates = None
    packed_coordinates = row.get("coordinates")
    if isinstance(packed_coordinates, dict):
        coordinates = GenomicCoordinates(**packed_coordinates)
    elif all(name in row and pd.notna(row[name]) for name in ("chrom", "chrom_start", "chrom_end", "strand")):
        coordinates = GenomicCoordinates(str(row["chrom"]), int(row["chrom_start"]), int(row["chrom_end"]), str(row["strand"]))
    stored = row.get("stored_raw_design_score", row.get("raw_design_score"))
    return Candidate(
        candidate_id=_row_id(row), rbd5=str(row["rbd5"]), rbd3=str(row["rbd3"]),
        provenance=Provenance("loaded", dataset_id, imported_from=dataset_id),
        gene_id=row.get("gene", row.get("gene_name")), transcript_id=row.get("transcript_id"),
        junction_id=row.get("junction_id"), coordinates=coordinates,
        on_junction=_optional_bool(row.get("on_junction")),
        junction_spanning=_optional_bool(row.get("junction_spanning")),
        stored_raw_design_score=float(stored) if pd.notna(stored) else None,
    )


def _optional_bool(value):
    return None if value is None or pd.isna(value) else bool(value)